# Framework RMIA with WRN-28-2 (CIFAR-10) on Colab
Runs `run_rmia.py` with `configs/rmia_cifar10_wrn28_2.json` (18 models = 9 complementary pairs, 4 targets, gamma 1/2, R = 1/2/4/8). Use a GPU runtime (T4 is fine). Results and per-model cache go to Google Drive so a disconnect does not lose finished models.

**Untested on Colab** (written without access). Do the smoke test first.


In [ ]:
import subprocess; print(subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout or 'NO GPU')
from google.colab import drive; drive.mount('/content/drive')


In [ ]:
%%bash
# Code: upload framework_colab.zip (or framework.zip) to /content first. The GitHub repo does NOT yet contain the WRN files, so cloning is only a fallback.
set -e
cd /content
rm -rf framework_clone
if ls framework*.zip >/dev/null 2>&1; then rm -rf framework; unzip -q -o $(ls framework*.zip | head -1); fi
[ -d framework ] || { echo "NO framework folder: upload framework_colab.zip to /content"; exit 1; }
test -f framework/configs/rmia_cifar10_wrn28_2.json || { echo "zip is missing the WRN config"; exit 1; }
cd framework && pip install -q -r requirements.txt 2>/dev/null || true
ls


In [ ]:
# Smoke test: 1 pair, 2 epochs, to check the GPU path (a few minutes). Then run the full cell below.
import json, subprocess
c = json.load(open('/content/framework/configs/rmia_cifar10_wrn28_2.json'))
c['name'] = 'rmia_wrn_smoke'; c['train']['epochs'] = 2; c['models']['num_pairs'] = 1
c['attack']['targets'] = [0]; c['attack']['num_ref'] = [1]
json.dump(c, open('/content/framework/configs/rmia_wrn_smoke.json', 'w'))
r = subprocess.run('cd /content/framework && python -u run_rmia.py --config configs/rmia_wrn_smoke.json --root /content/smoke_runs', shell=True, capture_output=True, text=True); print(r.stdout[-2500:], r.stderr[-2500:])


In [ ]:
# Full run: resumable, cache on Drive. Re-run this cell after a disconnect.
import os, subprocess
ROOT = '/content/drive/MyDrive/dp_rmia_runs'; os.makedirs(ROOT, exist_ok=True)
p = subprocess.Popen(f'cd /content/framework && python -u run_rmia.py --config configs/rmia_cifar10_wrn28_2.json --root {ROOT} 2>&1 | tee {ROOT}/rmia_wrn.log', shell=True)
p.wait()


In [ ]:
print(open('/content/drive/MyDrive/dp_rmia_runs/rmia_wrn.log').read()[-2500:])
